In [3]:
import pandas as pd
from pathlib import Path    
from statsmodels.stats.multitest import multipletests
from scipy import stats
import numpy as np

In [4]:
result_path = Path("../results/downstream_task")
# methods_list = ["fw-mrs-temperature-mean", "fw-mrs-temperature", "kmm", "uniform", "psa", "mrs-forest"]
methods_list = ["fw-mrs-temperature", "kmm", "uniform", "psa", "mrs-forest"]
# bias_types = ["less_negative_class", "less_positive_class", "mean_difference"]
bias_types = ["less_positive_class"]
metrics = ["AUROC", "AUPRC"]
bias_strengths = ["0.1"]
datasets = ["breast_cancer", "folktables_employment", "folktables_income", "hr_analytics", "loan_prediction"]

In [5]:
method_pairs = []
for j in range(1, len(methods_list)):
    method_pairs.append((methods_list[0], methods_list[j]))
method_pairs

[('fw-mrs-temperature', 'kmm'),
 ('fw-mrs-temperature', 'uniform'),
 ('fw-mrs-temperature', 'psa'),
 ('fw-mrs-temperature', 'mrs-forest')]

In [6]:
aurocs = []
auprcs = []
dict_list = []
for dataset in datasets:
    for bias_type in bias_types:
        for method in methods_list:
            for bias_strength in bias_strengths:
                json_directory = result_path / dataset / bias_type /  bias_strength/ method / "classification_results"
                auroc_file = pd.read_json(str(json_directory / "rf_auroc.json"))
                auprc_file = pd.read_json(str(json_directory / "rf_auprc.json"))
                dict_list.append({"Method": method, "Data Set": dataset, "AUROC": auroc_file.values, "AUPRC":auprc_file.values, 
                                  "Bias Type": bias_type, "Bias Strength": bias_strength})
result_df = pd.DataFrame(data=dict_list)

In [7]:
def corrected_t_test(first_values, second_values):
    differences = first_values - second_values
    mean_differences = np.mean(differences)
    variance_differences = np.var(differences)
    corrected_variance = variance_differences / len(first_values)
    correction_factor = 1.0 + ((2.0 / 3.0) / (1.0 / 3.0))
    return mean_differences / (np.sqrt(corrected_variance * correction_factor))

In [16]:
p_values = []
for dataset in datasets:
    for bias_type in bias_types:
            for bias_strength in bias_strengths:
                for first_method_name, second_metric_name in method_pairs:
                    for metric in metrics:
                        first_metrics = result_df.loc[(result_df["Method"]==first_method_name) & (result_df["Bias Type"]==bias_type) & 
                                                    (result_df["Bias Strength"]==bias_strength) & (result_df["Data Set"]==dataset)][metric].values[0]
                        second_metrics = result_df.loc[(result_df["Method"]==second_metric_name) & (result_df["Bias Type"]==bias_type) & 
                                                    (result_df["Bias Strength"]==bias_strength) & (result_df["Data Set"]==dataset)][metric].values[0]
                        t_statistic = corrected_t_test(first_metrics, second_metrics)
                        p_values.append(stats.t.sf(np.abs(t_statistic), len(first_metrics-1)) * 2)
corrected_p_values = multipletests(p_values, method="fdr_bh")

In [21]:
i = 0
for dataset in datasets:
    for bias_type in bias_types:
            for bias_strength in bias_strengths:
                for first_method_name, second_metric_name in method_pairs:
                    for metric in metrics:
                        print(f"p value for {dataset}, {bias_type}, {bias_strength}, {first_method_name}, {second_metric_name} is: {corrected_p_values[0][i]}")
                        i += 1

p value for breast_cancer, less_positive_class, 0.1, fw-mrs-temperature, kmm is: False
p value for breast_cancer, less_positive_class, 0.1, fw-mrs-temperature, kmm is: False
p value for breast_cancer, less_positive_class, 0.1, fw-mrs-temperature, uniform is: False
p value for breast_cancer, less_positive_class, 0.1, fw-mrs-temperature, uniform is: False
p value for breast_cancer, less_positive_class, 0.1, fw-mrs-temperature, psa is: False
p value for breast_cancer, less_positive_class, 0.1, fw-mrs-temperature, psa is: False
p value for breast_cancer, less_positive_class, 0.1, fw-mrs-temperature, mrs-forest is: False
p value for breast_cancer, less_positive_class, 0.1, fw-mrs-temperature, mrs-forest is: False
p value for folktables_employment, less_positive_class, 0.1, fw-mrs-temperature, kmm is: True
p value for folktables_employment, less_positive_class, 0.1, fw-mrs-temperature, kmm is: True
p value for folktables_employment, less_positive_class, 0.1, fw-mrs-temperature, uniform is: T

In [83]:
result_df["Mean AUROC"] = [np.mean(result_df["AUROC"].values[i], axis=0)[0] for i in range(len(result_df))]
result_df["Rank AUROC"] = result_df.groupby("Data Set")["Mean AUROC"].rank(ascending=False)
result_df["Mean AUPRC"] = [np.mean(result_df["AUPRC"].values[i], axis=0)[0] for i in range(len(result_df))]
result_df["Rank AUPRC"] = result_df.groupby("Data Set")["Mean AUPRC"].rank(ascending=False)

In [84]:
result_df[["Method", "Rank AUPRC", "Rank AUROC"]].groupby("Method").mean()

,Rank AUPRC,Rank AUROC
Method,,
fw-mrs-temperature,3.4,3.4
kmm,4.2,4.2
mrs-forest,1.6,1.8
psa,3.2,3.2
uniform,2.6,2.4
